# Анализ стартап-экосистемы

Exploratory Data Analysis и визуализация данных о венчурных инвестициях, компаниях и сделках M&A.

**Инструменты:** Python, pandas, matplotlib, seaborn, plotly

**Источник данных:** CSV из папки `data/` (схема совпадает с [SQL-проектом](../SQL/))

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns

DATA_DIR = Path("../data")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

## 1. Загрузка данных

In [ ]:
company = pd.read_csv(DATA_DIR / "company.csv", parse_dates=["founded_at", "closed_at"])
funding_round = pd.read_csv(DATA_DIR / "funding_round.csv", parse_dates=["funded_at"])
acquisition = pd.read_csv(DATA_DIR / "acquisition.csv", parse_dates=["acquired_at"])
fund = pd.read_csv(DATA_DIR / "fund.csv", parse_dates=["founded_at"])
investment = pd.read_csv(DATA_DIR / "investment.csv")

print(f"Компаний: {len(company)}")
print(f"Раундов финансирования: {len(funding_round)}")
print(f"Сделок M&A: {len(acquisition)}")
print(f"Фондов: {len(fund)}")
company.head()

## 2. Общая статистика

In [ ]:
summary = pd.DataFrame({
    "Метрика": [
        "Всего компаний",
        "Суммарное финансирование, $M",
        "Средний раунд, $M",
        "Сделок M&A",
        "Доля закрытых компаний, %",
    ],
    "Значение": [
        len(company),
        round(company["funding_total"].sum() / 1e6, 1),
        round(funding_round["raised_amount"].mean() / 1e6, 2),
        len(acquisition),
        round((company["status"] == "closed").mean() * 100, 1),
    ],
})
summary

## 3. Финансирование по странам

In [ ]:
by_country = (
    company.groupby("country_code", as_index=False)["funding_total"]
    .sum()
    .sort_values("funding_total", ascending=False)
    .head(10)
)
by_country["funding_total_m"] = by_country["funding_total"] / 1e6

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=by_country, x="country_code", y="funding_total_m", ax=ax, hue="country_code", legend=False)
ax.set_title("Топ-10 стран по объёму финансирования стартапов")
ax.set_xlabel("Страна")
ax.set_ylabel("Финансирование, млн $")
plt.tight_layout()
plt.show()

## 4. Распределение компаний по статусу и категории

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

status_counts = company["status"].value_counts()
axes[0].pie(status_counts, labels=status_counts.index, autopct="%1.1f%%", startangle=90)
axes[0].set_title("Статус компаний")

category_top = company["category_code"].value_counts().head(8)
sns.barplot(x=category_top.values, y=category_top.index, ax=axes[1], hue=category_top.index, legend=False)
axes[1].set_title("Топ категорий стартапов")
axes[1].set_xlabel("Количество")
plt.tight_layout()
plt.show()

## 5. Динамика раундов финансирования

In [ ]:
funding_round["year"] = funding_round["funded_at"].dt.year
by_year = (
    funding_round.groupby("year", as_index=False)
    .agg(rounds=("id", "count"), total_raised=("raised_amount", "sum"))
)
by_year["total_raised_m"] = by_year["total_raised"] / 1e6

fig, ax1 = plt.subplots(figsize=(11, 5))
color1 = "#4C72B0"
ax1.bar(by_year["year"], by_year["rounds"], color=color1, alpha=0.7, label="Число раундов")
ax1.set_xlabel("Год")
ax1.set_ylabel("Число раундов", color=color1)

ax2 = ax1.twinx()
color2 = "#C44E52"
ax2.plot(by_year["year"], by_year["total_raised_m"], color=color2, marker="o", linewidth=2, label="Сумма, млн $")
ax2.set_ylabel("Сумма привлечённых средств, млн $", color=color2)
ax1.set_title("Динамика венчурных раундов по годам")
plt.tight_layout()
plt.show()

## 6. Активность венчурных фондов

In [ ]:
def fund_activity(n: int) -> str:
    if n >= 100:
        return "high_activity"
    if n >= 20:
        return "middle_activity"
    return "low_activity"

fund["activity"] = fund["invested_companies"].apply(fund_activity)
activity_stats = fund.groupby("activity")["investment_rounds"].mean().reset_index()

fig = px.bar(
    activity_stats,
    x="activity",
    y="investment_rounds",
    title="Среднее число инвестиционных раундов по сегменту активности фондов",
    labels={"activity": "Сегмент", "investment_rounds": "Среднее число раундов"},
    color="activity",
)
fig.show()

## 7. Сделки M&A: сумма и тип оплаты

In [ ]:
acquisition["year"] = acquisition["acquired_at"].dt.year
mna_by_term = acquisition.groupby("term_code")["price_amount"].sum().reset_index()
mna_by_term["price_amount_m"] = mna_by_term["price_amount"] / 1e6

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=mna_by_term, x="term_code", y="price_amount_m", ax=axes[0], hue="term_code", legend=False)
axes[0].set_title("Сумма сделок M&A по типу оплаты")
axes[0].set_ylabel("Сумма, млн $")

mna_yearly = acquisition.groupby("year")["price_amount"].sum() / 1e6
axes[1].plot(mna_yearly.index, mna_yearly.values, marker="s", color="#55A868")
axes[1].set_title("Объём сделок M&A по годам")
axes[1].set_xlabel("Год")
axes[1].set_ylabel("Сумма, млн $")
plt.tight_layout()
plt.show()

## 8. Интерактивная карта: финансирование по странам

In [ ]:
country_map = {
    "USA": "United States", "GBR": "United Kingdom", "DEU": "Germany",
    "FRA": "France", "ISR": "Israel", "CAN": "Canada", "IND": "India",
    "CHN": "China", "JPN": "Japan", "BRA": "Brazil",
}
map_data = by_country.copy()
map_data["country"] = map_data["country_code"].map(country_map)

fig = px.choropleth(
    map_data,
    locations="country",
    locationmode="country names",
    color="funding_total_m",
    title="География финансирования стартапов (млн $)",
    color_continuous_scale="Blues",
)
fig.show()

## 9. Выводы

- **География:** основной объём финансирования сосредоточен в USA, GBR, DEU и ISR.
- **Статус:** большинство компаний продолжают работать; доля закрытых — показатель риска экосистемы.
- **Раунды:** динамика инвестиций по годам позволяет выявить периоды активности рынка.
- **Фонды:** сегментация по `invested_companies` показывает различие в активности high/middle/low фондов.
- **M&A:** сделки с оплатой cash доминируют по объёму; помесячная/годовая динамика отражает циклы поглощений.

Данные и SQL-запросы к той же предметной области — в разделах [data/](../data/) и [SQL/](../SQL/).